In [0]:

# AlertaBairro - Camada Bronze
# Etapa 1: Verificar arquivos de entrada

caminho_volume = "/Volumes/workspace/alertabairro/chegada/"

arquivos = dbutils.fs.ls(caminho_volume)

print("=== ARQUIVOS DISPONIVEIS ===")

for arquivo in arquivos:
    print(f"Nome: {arquivo.name}")
    print(f"Tamanho: {arquivo.size} bytes")
    print("---------------------------")

print(f"Total de arquivos: {len(arquivos)}")


=== ARQUIVOS DISPONIVEIS ===
Nome: alagamentos_20261111.geojson
Tamanho: 163265 bytes
---------------------------
Nome: clima_20261111.json
Tamanho: 621 bytes
---------------------------
Total de arquivos: 2


In [0]:

# AlertaBairro - Camada Bronze
# Etapa 2: Leitura dos dados com PySpark

from pyspark.sql.functions import size

# Caminhos dos arquivos
caminho_clima = (
    "/Volumes/workspace/alertabairro/chegada/"
    "clima_20261111.json"
)

caminho_alagamentos = (
    "/Volumes/workspace/alertabairro/chegada/"
    "alagamentos_20261111.geojson"
)

# Leitura dos arquivos JSON
df_clima = (
    spark.read
    .option("multiLine", "true")
    .json(caminho_clima)
)

df_alagamentos = (
    spark.read
    .option("multiLine", "true")
    .json(caminho_alagamentos)
)

# Verificar os dados climáticos
print("=== DADOS CLIMATICOS ===")
df_clima.printSchema()

df_clima.select(
    size("daily.time").alias("total_dias")
).show()

# Verificar os dados de alagamento
print("=== DADOS DE ALAGAMENTOS ===")
df_alagamentos.printSchema()

df_alagamentos.select(
    size("features").alias("total_ocorrencias")
).show()


=== DADOS CLIMATICOS ===
root
 |-- daily: struct (nullable = true)
 |    |-- precipitation_sum: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- daily_units: struct (nullable = true)
 |    |-- precipitation_sum: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (nullable = true)

+----------+
|total_dias|
+----------+
|         8|
+----------+

=== DADOS DE ALAGAMENTOS ===
root
 |-- crs: struct (nullable = true)
 |    |-- properties: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- type: string (nullable = true)
 |-- features:

In [0]:

# AlertaBairro - Camada Bronze
# Etapa 3: Criar tabelas Delta

spark.sql("""
    CREATE TABLE IF NOT EXISTS
    workspace.alertabairro.bronze_clima
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS
    workspace.alertabairro.bronze_alagamentos
""")

print("=== TABELAS BRONZE CRIADAS ===")

spark.sql("""
    SHOW TABLES IN workspace.alertabairro
""").show(truncate=False)


=== TABELAS BRONZE CRIADAS ===
+------------+------------------+-----------+
|database    |tableName         |isTemporary|
+------------+------------------+-----------+
|alertabairro|bronze_alagamentos|false      |
|alertabairro|bronze_clima      |false      |
+------------+------------------+-----------+



In [0]:

# AlertaBairro - Camada Bronze
# Etapa 4: Ingestao incremental dos dados climaticos

resultado = spark.sql("""
    COPY INTO workspace.alertabairro.bronze_clima
    FROM '/Volumes/workspace/alertabairro/chegada/'
    FILEFORMAT = JSON
    PATTERN = 'clima_*.json'
    FORMAT_OPTIONS (
        'multiLine' = 'true',
        'mergeSchema' = 'true'
    )
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

print("=== RESULTADO DA INGESTAO ===")
resultado.show(truncate=False)

print("=== DADOS NA BRONZE ===")
spark.sql("""
    SELECT
        COUNT(*) AS documentos_json,
        SUM(size(daily.time)) AS total_dias
    FROM workspace.alertabairro.bronze_clima
""").show()


=== RESULTADO DA INGESTAO ===
+-----------------+-----------------+-------------------------+
|num_affected_rows|num_inserted_rows|num_skipped_corrupt_files|
+-----------------+-----------------+-------------------------+
|0                |0                |0                        |
+-----------------+-----------------+-------------------------+

=== DADOS NA BRONZE ===
+---------------+----------+
|documentos_json|total_dias|
+---------------+----------+
|              2|        16|
+---------------+----------+



In [0]:

# AlertaBairro - Camada Bronze
# Etapa 5: Ingestao incremental dos alagamentos

resultado = spark.sql("""
    COPY INTO workspace.alertabairro.bronze_alagamentos
    FROM '/Volumes/workspace/alertabairro/chegada/'
    FILEFORMAT = JSON
    PATTERN = 'alagamentos_*.geojson'
    FORMAT_OPTIONS (
        'multiLine' = 'true',
        'mergeSchema' = 'true'
    )
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

print("=== RESULTADO DA INGESTAO ===")
resultado.show(truncate=False)

print("=== DADOS NA BRONZE ===")
spark.sql("""
    SELECT
        COUNT(*) AS documentos_geojson,
        SUM(size(features)) AS total_ocorrencias
    FROM workspace.alertabairro.bronze_alagamentos
""").show()


=== RESULTADO DA INGESTAO ===
+-----------------+-----------------+-------------------------+
|num_affected_rows|num_inserted_rows|num_skipped_corrupt_files|
+-----------------+-----------------+-------------------------+
|0                |0                |0                        |
+-----------------+-----------------+-------------------------+

=== DADOS NA BRONZE ===
+------------------+-----------------+
|documentos_geojson|total_ocorrencias|
+------------------+-----------------+
|                 2|              584|
+------------------+-----------------+



In [0]:

%sql
SELECT
    'Clima' AS fonte,
    COUNT(*) AS total_documentos
FROM workspace.alertabairro.bronze_clima

UNION ALL

SELECT
    'Alagamentos' AS fonte,
    COUNT(*) AS total_documentos
FROM workspace.alertabairro.bronze_alagamentos;


fonte,total_documentos
Clima,2
Alagamentos,2
